# Week 3 — Hypothesis testing, preparing data, and correlation

Three things this week, in the order you would actually do them.

1. **Hypothesis testing** — the reasoning every test in the rest of the module
   uses.
2. **Preparing data** — deciding whether your variables can carry the test you
   want to run.
3. **Correlation** — the first real test, and the first real report.

In [ ]:
import math9102 as m9
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

m9.use_house_style()

survey = m9.load_survey()

---

## Part 1 — Hypothesis testing

### The statistical model

Every model in this module has the same shape:

$$\text{outcome}_i = \text{model} + \text{error}_i$$

For a straight line, the model is $b_0 + b_1 x$, and the error is how far each
point sits from the line. The whole business of statistics is choosing the model
and then deciding whether the pattern it found is more than noise.

### What a hypothesis test is

A method that uses sample data to decide between two competing claims about a
population.

> **H₀ (the null hypothesis)** — there is no effect, no difference, no
> relationship. Always contains an equality.
>
> **H₁ (the alternative)** — the competing claim. Always contains an inequality.
> This is what your theory predicts.

You start by *assuming* H₀. Then you ask: **if H₀ were true, how surprising
would my data be?**

### The p-value

> The **p-value** is the probability of observing a test statistic at least as
> extreme as the one you got, **assuming the null hypothesis is true**.

Every clause matters. It is a probability about *data given a hypothesis*, not
about a hypothesis given data.

In [ ]:
observed = 1.8

fig, ax = plt.subplots(figsize=(8, 3))
x = np.linspace(-4, 4, 600)
ax.plot(x, stats.norm.pdf(x))
for lo, hi in [(observed, 4), (-4, -observed)]:
    mask = (x >= lo) & (x <= hi)
    ax.fill_between(x[mask], stats.norm.pdf(x[mask]), alpha=0.4)
ax.set_title(f"two-tailed p = {2 * stats.norm.sf(observed):.4f}")
ax.set_yticks([]);

### Choosing α

α is the threshold you set **in advance**. It is the probability you are willing
to accept of rejecting a true null hypothesis.

In [ ]:
for alpha in (0.05, 0.01, 0.001):
    print(f"alpha = {alpha:<6} -> {100 * (1 - alpha):.1f}% confidence interval, "
          f"critical z = {stats.norm.ppf(1 - alpha / 2):.3f}")

A confidence interval is the same idea from the other side. A 95% interval means
that if you repeated the study many times, 95% of the intervals you constructed
would contain the true population value. **It is a statement about the
procedure, not about this particular interval.**

### Two errors, and power

|  | H₀ is actually true | H₀ is actually false |
|---|---|---|
| **You reject H₀** | **Type I error** (probability α) | Correct — this is *power* |
| **You do not reject H₀** | Correct | **Type II error** (probability β) |

**Power = 1 − β**: the probability of detecting an effect that is really there.
Power rises with the sample size and with the size of the effect.

Note what the table shows about α: **you choose your Type I error rate**. Lower
it to 0.01 and you make fewer false positives — and, holding everything else
constant, more false negatives.

### Two conclusions you must not draw

In [ ]:
m9.md("""
- A **non-significant** result does not mean H₀ is true and does not prove it.
  It means this sample did not provide sufficient evidence against it.
- A **significant** result does not mean H₁ is true and does not prove it. It
  means this sample provided sufficient evidence to reject H₀ in its favour.
""")

And the caveat that follows from taking α seriously: is a p-value of .049 really
a different finding from .051? **No.** The threshold is a convention that makes
decisions repeatable, not a line in nature. Report the actual value, report an
effect size, and let the reader see how close it was.

---

## Part 2 — Preparing data

### Parametric or non-parametric?

| | Parametric | Non-parametric |
|---|---|---|
| Assumes a distribution for the population | yes — usually normal | no |
| Works on | interval / ratio | ordinal, or badly skewed, or small samples |
| Power, when its assumptions hold | higher | lower |

So the decision procedure is: **look at the variable, decide whether the normal
model is close enough, and say what you decided from.**

### How to inspect a continuous variable

In [ ]:
m9.normality_panel(survey.tpcoiss.dropna(), label="Total perceived control")

The Q-Q plot is the one to learn to read. It plots your data's quantiles against
the quantiles a normal distribution would have produced. If the two agree, the
points lie on the line.

In [ ]:
rng = np.random.default_rng(6)
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (sample, name) in zip(axes, [
    (rng.normal(0, 1, 400), "normal"),
    (rng.gamma(1.6, 1, 400), "right-skewed"),
    (-rng.gamma(1.6, 1, 400), "left-skewed"),
    (rng.standard_t(3, 400), "heavy tails"),
]):
    m9.qq_plot(sample, ax=ax, title=name)
fig.tight_layout();

Four pictures. Learn them and you can read any Q-Q plot you meet.

### Standardised skew and kurtosis, and why the threshold fails

You will meet a heuristic: divide skew by its standard error and compare against
±1.96. Here it is on this dataset.

In [ ]:
x = survey.tpcoiss.dropna()
n = len(x)

skew = stats.skew(x, bias=False)
se_skew = np.sqrt(6 * n * (n - 1) / ((n - 2) * (n + 1) * (n + 3)))

print(f"n = {n}")
print(f"skew = {skew:.4f}")
print(f"standard error of skew = {se_skew:.4f}")
print(f"standardised skew = {skew / se_skew:.2f}")

Now look at what that standard error is made of. **It depends only on the sample
size.** The same shape, measured on a bigger sample, produces a bigger ratio.

In [ ]:
for size in (30, 80, 200, 430, 1000, 5000):
    se = np.sqrt(6 * size * (size - 1) / ((size - 2) * (size + 1) * (size + 3)))
    print(f"n = {size:5d}: a skew of {skew:.3f} gives a ratio of {skew / se:6.2f}")

The distribution's shape has not changed once in that table. Only the verdict
has.

**This is why the module does not gate on the ratio.** Report it as a
description of shape with the sample size beside it, and let the plots decide.

In [ ]:
m9.report_normality(survey, "tpcoiss", "Total perceived control")

In [ ]:
m9.report_normality(survey, "tpstress", "Total perceived stress")

The same argument applies to Shapiro–Wilk and Kolmogorov–Smirnov, which are
goodness-of-fit tests with the same sample-size sensitivity:

In [ ]:
for size in (30, 100, 200, len(x)):
    sub = x.sample(size, random_state=1)
    print(f"n = {size:4d}: Shapiro-Wilk p = {stats.shapiro(sub).pvalue:.4f}")

One variable, four nested samples of it, and the verdict tightens as the sample
grows. **The test is telling you about your sample size at least as much as
about your distribution.**

### Missing data

Three mechanisms, and the difference between them decides what you may do.

| Mechanism | Meaning | Can you ignore it? |
|---|---|---|
| **MCAR** — missing completely at random | missingness is unrelated to anything, observed or not | yes |
| **MAR** — missing at random | missingness depends on variables you *did* observe | sometimes, if you model it |
| **MNAR** — missing not at random | missingness depends on the missing value itself | **no** |

The names are unhelpful: "missing at random" does not mean random. MNAR is the
dangerous one, and **you cannot test for it**, because the evidence you would
need is exactly the data you do not have.

In [ ]:
scales = ["toptim", "tmast", "tposaff", "tnegaff", "tlifesat",
          "tpstress", "tslfest", "tmarlow", "tpcoiss"]

survey[scales].isna().sum().sort_values(ascending=False)

In [ ]:
everything = m9.missingness(survey, scales)
print(everything.attrs["summary"])
everything

In [ ]:
two = m9.missingness(survey, ["tpcoiss", "tpstress"])
print(two.attrs["summary"])

Look for **pattern**, not just amount. Is missingness concentrated in one
variable, or spread evenly? Is it related to a group?

In [ ]:
by_child = (survey.assign(missing_control=survey.tpcoiss.isna())
                  .groupby("child").missing_control.mean())
by_child.round(4)

**A useful benchmark.** Tabachnick and Fidell: if missing data is under about
5% of the total and appears random in a large dataset, almost any treatment
gives similar results. Above that, the treatment starts to matter.

### What to do about it

- **Listwise deletion** (complete cases). Simple, and the remaining data is
  complete — at the cost of sample size and, if the data are not MCAR, of bias.
- **Pairwise deletion**. Uses each case wherever it has the needed values, so
  the sample size varies between analyses, which complicates standard errors.
- **Imputation** — mean substitution, regression prediction, nearest neighbour.
  Cheap and biased: mean substitution shrinks the variance and weakens
  correlations by construction.
- **Modern methods** — maximum likelihood and multiple imputation. These give
  unbiased estimates and honest standard errors, and are what you should reach
  for on a real project.

Whatever you do, **report it**. The treatment is part of the method.

### Bias

Missing data is one route to bias. There are others, and they are all the same
problem: the sample differs *systematically* from the population.

- **Selection bias** — using only certain cases or groups.
- **Sampling bias** — non-random sampling. A fact of life with convenience
  samples; recognise it and say so.
- **Non-response bias** — the people who did not answer differ from those who
  did.
- **Time-interval bias** — collecting over a window that is not representative.
- **Confirmation bias** — choosing data that supports what you already believe.
- **Omitted-variable bias** — leaving a relevant concept out of the model.

None of these is fixed by a larger sample.

### Outliers, again

Week 2 covered the IQR fence. A second screen, useful for interval and ratio
variables, is the standardised score.

In [ ]:
z = (x - x.mean()) / x.std(ddof=1)
threshold = 3.29 if len(x) > 80 else 2.5

print(f"n = {len(x)}, so the threshold is {threshold}")
print(f"cases beyond it: {(z.abs() > threshold).sum()} "
      f"({(z.abs() > threshold).mean():.2%})")

Note the two thresholds: ±2.5 for samples of 80 or fewer, ±3.29 above that. And
note what a **multivariate** outlier is — a case whose value on each variable is
unremarkable but whose *combination* is rare. A person who is 150 cm tall is
ordinary; a person who is 150 cm tall and weighs 140 kg is not, and neither
variable alone would flag them.

### Transformation

If a variable is badly skewed you can transform it and re-check.

In [ ]:
for name, transformed in [
    ("raw", survey.tnegaff.dropna()),
    ("square root", np.sqrt(survey.tnegaff.dropna())),
    ("log(x + 1)", np.log(survey.tnegaff.dropna() + 1)),
    ("reciprocal", 1 / (survey.tnegaff.dropna() + 1)),
]:
    print(f"{name:14s} skew = {stats.skew(transformed, bias=False):+.3f}")

Three warnings.

1. **Check that it worked.** A transformation that does not improve the shape is
   one you do not apply.
2. **You are now interpreting the transformed variable.** A coefficient on
   `log(income)` is not a coefficient on income.
3. **Some values are not permissible** — the log of zero, division by zero, the
   square root of a negative. Decide what to do about them, and say so.

If transformation does not help, use a test that does not need normality.

---

## Part 3 — Correlation

### From covariance to correlation

Variance measures how much one variable deviates from its own mean.
**Covariance** does the same for two variables at once: multiply each case's
deviation on *x* by its deviation on *y*, and average.

In [ ]:
pairs = survey[["tpcoiss", "tpstress"]].dropna()

dx = pairs.tpcoiss - pairs.tpcoiss.mean()
dy = pairs.tpstress - pairs.tpstress.mean()
covariance = (dx * dy).sum() / (len(pairs) - 1)

print(f"covariance = {covariance:.3f}")
print(f"pandas agrees: {pairs.tpcoiss.cov(pairs.tpstress):.3f}")

Positive covariance means the two rise together; negative means one rises as the
other falls. But the size is uninterpretable, because it depends on the units.
Measure the same thing in kilometres instead of miles and the covariance
changes.

**Correlation is standardised covariance.** Divide by both standard deviations
and it is forced between −1 and +1:

In [ ]:
r_by_hand = covariance / (pairs.tpcoiss.std(ddof=1) * pairs.tpstress.std(ddof=1))
print(f"r = {r_by_hand:.6f}")

### Plot it first

In [ ]:
m9.scatter_with_fit(survey, "tpcoiss", "tpstress",
                    xlabel="Total perceived control of internal states",
                    ylabel="Total perceived stress");

Read three things off a scatterplot before computing anything:

- **Direction** — does the cloud slope up or down?
- **Strength** — how tightly do the points hug the line?
- **Form** — is it a line at all?

The reason to look first is Anscombe's quartet: four datasets with the same
mean, the same variance and the same correlation, and nothing else in common.

In [ ]:
x1 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
quartet = [
    (x1, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    (x1, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    (x1, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04,
                                          5.25, 12.50, 5.56, 7.91, 6.89]),
]

fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharex=True, sharey=True)
for ax, (xs, ys) in zip(axes, quartet):
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys)
    ax.scatter(xs, ys)
    ax.set_title(f"r = {np.corrcoef(xs, ys)[0, 1]:.2f}")
fig.tight_layout();

**Only the first one is a correlation worth reporting.**

### Assumptions of Pearson's r

1. **Two interval or ratio variables.** (One may be binary, with roughly equal
   group sizes.)
2. **Related pairs** — each case supplies a value on both.
3. **Independent observations** — no case influences another.
4. **Normality** — check each variable.
5. **Linearity** — check the scatterplot. Pearson measures straight-line
   association and nothing else.
6. **Homoscedasticity** — the spread around the line should be roughly constant.
   A tube, not a cone.

Homoscedasticity is the one people skip, so here is why it matters:

In [ ]:
rng = np.random.default_rng(8)
income = rng.uniform(20, 120, 400)
cone = 0.35 * income + rng.normal(0, 1, 400) * (0.06 * income)
tube = 0.35 * income + rng.normal(0, 4.2, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, y, name in [(axes[0], cone, "heteroscedastic"), (axes[1], tube, "homoscedastic")]:
    ax.scatter(income, y, s=10, alpha=0.6)
    ax.set_title(f"{name}: r = {np.corrcoef(income, y)[0, 1]:.2f}")
    ax.set_xlabel("income")
fig.tight_layout();

Same correlation. In the left panel the error grows with income, so a prediction
for a high earner is far less reliable than one for a low earner — and a single
correlation coefficient cannot tell you that.

### Run it

In [ ]:
result = stats.pearsonr(pairs.tpcoiss, pairs.tpstress)
result

In [ ]:
print(f"r = {result.statistic:.4f}")
print(f"p = {result.pvalue:.3g}")
print(f"95% CI = {result.confidence_interval(0.95)}")
print(f"n = {len(pairs)}")

Why is a *t*-statistic involved? Because the test of "is this correlation
different from zero?" is a *t*-test, on *n* − 2 degrees of freedom:

In [ ]:
r = result.statistic
n = len(pairs)
t = r * np.sqrt((n - 2) / (1 - r ** 2))

print(f"t = {t:.4f}, df = {n - 2}")
print(f"p from that t = {2 * stats.t.sf(abs(t), n - 2):.3g}")

**Degrees of freedom are not the sample size.** Report *n* as *n*, and *df* as
*df*; they differ by two here, and quoting one as the other is a common slip.

### r is an effect size

Unlike most test statistics, *r* is directly interpretable. Ignore the sign for
magnitude, and use Cohen's conventions:

In [ ]:
for value in (0.1, 0.3, 0.5, result.statistic):
    print(f"r = {value:+.3f} -> {m9.interpret(value, 'r')}")

Squaring it gives the **coefficient of determination**, the proportion of
variance the two variables share:

In [ ]:
print(f"r squared = {result.statistic ** 2:.4f}")
print(f"shared variance = {result.statistic ** 2:.1%}")

### Statistical against practical significance

In [ ]:
rng = np.random.default_rng(3)
for size in (30, 100, 1000, 5000):
    a = rng.normal(0, 1, size)
    b = 0.1 * a + rng.normal(0, 1, size)
    res = stats.pearsonr(a, b)
    print(f"n = {size:5d}: r = {res.statistic:+.3f}, p = {res.pvalue:.4f}, "
          f"shared variance = {res.statistic ** 2:.2%}")

A correlation of about 0.1 explains roughly one per cent of the variance. At a
sample of thirty it is invisible; at five thousand it is highly significant. It
is the same negligible relationship throughout. **Report the coefficient and the
shared variance, not just the p-value.**

### Report it

In [ ]:
m9.report_correlation(
    survey, "tpcoiss", "tpstress", result,
    x_label="perceived control of internal states",
    y_label="perceived stress",
)

### When Pearson is not appropriate

**Spearman's rho** ranks both variables and correlates the ranks, so it measures
*monotonic* association — does one rise as the other rises, by any shape of
curve? **Kendall's tau** counts concordant and discordant pairs.

In [ ]:
x = np.linspace(0.2, 5.0, 120)
y = np.exp(x)

print(f"a perfect exponential relationship:")
print(f"  Pearson r = {stats.pearsonr(x, y).statistic:.3f}")
print(f"  Spearman rho = {stats.spearmanr(x, y).statistic:.3f}")

Pearson understates it because the relationship is not a straight line. Spearman
gets it exactly right, because the ranks *are* perfectly ordered.

On the real data:

In [ ]:
rho = stats.spearmanr(pairs.tpcoiss, pairs.tpstress)
tau = stats.kendalltau(pairs.tpcoiss, pairs.tpstress)

print(f"Pearson  r   = {result.statistic:+.4f}, p = {result.pvalue:.3g}")
print(f"Spearman rho = {rho.statistic:+.4f}, p = {rho.pvalue:.3g}")
print(f"Kendall  tau = {tau.statistic:+.4f}, p = {tau.pvalue:.3g}")

**About ties.** Questionnaire totals produce many tied values, and you may read
that this is a problem for Spearman.

In [ ]:
print(f"tied values in tpcoiss: {pairs.tpcoiss.duplicated().sum()} of {len(pairs)}")

It is not. Ties are handled by a correction to the normal approximation, applied
automatically. In older R code they produce a *warning* about exact p-values,
not an error. **Ties are not missing data**, and they are not a reason to switch
tests or to drop cases.

Choose Spearman or Kendall on their merits — Spearman is more widely reported,
Kendall is more robust and both usually lead to the same conclusion — and run
**one** of them. Running several and reporting the friendliest is fishing.

### Correlation is not causation

*r* measures association. It cannot tell you:

- which variable causes which,
- whether a third variable causes both,
- or whether the relationship is causal at all.

Week 1's spurious-correlation figure is the reminder: two independent random
walks correlate strongly because both trend. Nothing about the arithmetic knows
that.

---

## What to take from this week

- A p-value is the probability of data at least this extreme **given H₀**, and
  nothing else.
- α, β and power are three views of the same trade-off, and you choose α in
  advance.
- Normality is assessed **visually first**. Standardised ratios and normality
  tests scale with sample size and cannot make the decision for you.
- Missing data has a mechanism, and the mechanism decides the treatment. MNAR
  cannot be tested for.
- Plot before you correlate. Anscombe.
- Report *r*, its shared variance and *n* — and never a p-value on its own.
- Ties are ordinary. They are not missing data.

## Exercise

`exercise.md` asks you to inspect two variables, justify a test, run it and
report it — the full six-step procedure you will use from week 4 onwards.